# Example: Let's Build and Test a Trade Bot
In this example, we'll combinae several concepts from the course to build a simple trade bot that can make buy/sell/hold decisions based on moving averages, bandit problems and utility maximization.

> __Learning Objectives__
>
> By the end of this example, students will be able to:
> Three learning objectives go here.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include-tradebot-synthetic.jl")); # include the Include.jl file

For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [VLQuantitativeFinancePackage.jl documentation](https://github.com/varnerlab/VLQuantitativeFinancePackage.jl). 

### Data, Models and Constants
Before we implement the trade bot, we need to load up some models and data first. First, let's load the HMM model that we estimated using historical data (2014 - 2024). To start, we specify the `path_to_save_file::String` variable, and then we load the HMM model:

In [2]:
saved_state_dict = let

    # initialize -
    path_to_save_file = joinpath(_PATH_TO_DATA,"HMM-WJ-SPY-N-100-daily-aggregate.jld2");
    saved_state_dict = load(path_to_save_file);
    saved_state_dict; # return the loaded dictionary
end

┌ Warning: type VLQuantitativeFinancePackage.MyHiddenMarkovModel does not exist in workspace; reconstructing
└ @ JLD2 /Users/jdv27/.julia/packages/JLD2/SgtOb/src/data/reconstructing_datatypes.jl:588
┌ Warning: type VLQuantitativeFinancePackage.MyHiddenMarkovModelWithJumps does not exist in workspace; reconstructing
└ @ JLD2 /Users/jdv27/.julia/packages/JLD2/SgtOb/src/data/reconstructing_datatypes.jl:588


Dict{String, Any} with 11 entries:
  "risk_free_rate"          => 0.043
  "model"                   => Reconstruct@MyHiddenMarkovModel(Any[[1, 2, 3, 4,…
  "decode"                  => Dict{Int64, Normal}(5=>Normal{Float64}(μ=-3.3565…
  "encoded_archive_with_ju… => [53 72 … 82 92; 93 82 … 80 5; … ; 14 67 … 87 13;…
  "insampledataset"         => [-0.62754, 0.839626, 0.162992, -0.0111337, 0.257…
  "jump_model"              => Reconstruct@MyHiddenMarkovModelWithJumps(Any[[1,…
  "encoded_archive"         => [48 29 … 28 33; 42 10 … 10 68; … ; 18 37 … 39 77…
  "number_of_states"        => 100
  "in_sample_decoded_archi… => [0.202562 0.962389 … 1.56531 2.7557; 2.94573 1.6…
  "stationary"              => Categorical{Float64, Vector{Float64}}(…
  "in_sample_decoded_archi… => [0.0753184 -0.675607 … -0.759002 -0.470222; -0.1…

Next, let's load the single index model parameters that we computed in the previous example. We'll store this data in the `sim_model_parameters::Dict{String,NamedTuple}` variable. In addition, we return a few other useful variables, such as the historical market growth rate, the mean and variance of the market growth, etc.

In [3]:
sim_model_parameters,Gₘ,Ḡₘ,Varₘ = let

    # initialize -
    path_to_sim_model_parameters = joinpath(_PATH_TO_DATA,"SIMs-SPY-SP500-01-03-14-to-12-31-24.jld2");
    sim_model_parameters = JLD2.load(path_to_sim_model_parameters);
    parameters = sim_model_parameters["data"]; # return

    Gₘ = sim_model_parameters["Gₘ"]; # Get the past market growth rate 
    Ḡₘ = sim_model_parameters["Ḡₘ"]; # mean of market growth rates
    Varₘ = sim_model_parameters["Varₘ"]; # variance of market growth

    # return -
    parameters,  Gₘ , Ḡₘ, Varₘ;
end;

Let's define some constants and parameters that we will use in this example. See the comment lines next to each variable for a description of its purpose, units, permissible values, etc.

In [4]:
Δt = (1.0 / 252.0); # time step (in years) between trading days (assumes 252 trading days per year)
risk_free_rate = 0.043; # annualized risk-free interest rate (4.3% per year)
λ = 3.0; # risk-aversion parameter
number_of_training_steps = 10000; # number of training steps for the bandit agent

### Training Market Data

We gathered daily open-high-low-close (OHLC) data for each firm in the [S&P 500](https://en.wikipedia.org/wiki/S%26P_500) from `01-03-2014` until `12-31-2024`, along with data for several exchange-traded funds and volatility products during that time period.

Let's load the dataset by calling [the `MyTrainingMarketDataSet()` function](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/data/#VLQuantitativeFinancePackage.MyTrainingMarketDataSet).

In [5]:
original_dataset = MyTrainingMarketDataSet() |> x-> x["dataset"] # load the original dataset (training)

Dict{String, DataFrame} with 515 entries:
  "DD"   => 2329×8 DataFrame…
  "EMR"  => 2767×8 DataFrame…
  "CTAS" => 2767×8 DataFrame…
  "HSIC" => 2767×8 DataFrame…
  "KIM"  => 2767×8 DataFrame…
  "PLD"  => 2767×8 DataFrame…
  "IEX"  => 2767×8 DataFrame…
  "TPR"  => 1803×8 DataFrame…
  "BAC"  => 2767×8 DataFrame…
  "CBOE" => 2767×8 DataFrame…
  "EXR"  => 2767×8 DataFrame…
  "NCLH" => 2767×8 DataFrame…
  "CVS"  => 2767×8 DataFrame…
  "DRI"  => 2767×8 DataFrame…
  "DTE"  => 2767×8 DataFrame…
  "ZION" => 2767×8 DataFrame…
  "AVY"  => 2767×8 DataFrame…
  "EW"   => 2767×8 DataFrame…
  "EA"   => 2767×8 DataFrame…
  ⋮      => ⋮

Not all tickers in our dataset have the maximum number of trading days for various reasons, such as acquisition or delisting events. Let's collect only those tickers with the maximum number of trading days.

First, let's compute the number of records for a firm that we know has the maximum value, e.g., `AAPL`, and save that value in the `maximum_number_trading_days::Int64` variable:

In [6]:
maximum_number_trading_days = original_dataset["AAPL"] |> nrow; # maximum number of trading days in our dataset

Now, iterate through our data and collect only tickers with `maximum_number_trading_days` records. Save that data in the `dataset::Dict{String,DataFrame}` variable:

In [7]:
dataset = let

    # initialize -
    dataset = Dict{String, DataFrame}();

    # iterate through the dictionary; we can't guarantee a particular order
    for (ticker, data) ∈ original_dataset  # we get each (K, V) pair!
        if (nrow(data) == maximum_number_trading_days) # check if ticker has maximum trading days
            dataset[ticker] = data;
        end
    end
    dataset; # return
end;

Finally, let's get a list of the firms in our cleaned dataset and sort them alphabetically. We store the sorted firm ticker symbols in the `list_of_tickers_clean_price_data::Vector{String}` variable:

In [8]:
list_of_tickers_clean_price_data = keys(dataset) |> collect |> sort; # list of tickers in our clean dataset

Now let's get a list of all tickers for which we have single index model parameters:

In [9]:
tickers_that_we_have_sim_data_for = keys(sim_model_parameters) |> collect |> sort;

We need to use only the tickers for which we have both price data and SIM parameters. We'll compute [the intersection of the two lists](https://docs.julialang.org/en/v1/base/collections/#Base.intersect) and store the result in the `list_of_tickers::Vector{String}` variable:

In [10]:
list_of_tickers = intersect(tickers_that_we_have_sim_data_for, list_of_tickers_clean_price_data)

424-element Vector{String}:
 "A"
 "AAL"
 "AAP"
 "AAPL"
 "ABBV"
 "ABT"
 "ACN"
 "ADBE"
 "ADI"
 "ADM"
 ⋮
 "WYNN"
 "XEL"
 "XOM"
 "XRAY"
 "XYL"
 "YUM"
 "ZBRA"
 "ZION"
 "ZTS"

___

## Task 1: Initialize a world model and set up a ticker picker agent
Our first task is to construct models of the bandit agent and the world that this agent samples. We construct [a `MyEpsilonSamplingBanditModel` instance](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/bandits/#VLQuantitativeFinancePackage.MyEpsilonSamplingBanditModel) to hold agent data and [a `MyTickerPickerSIMRiskAwareWorldModel` instance](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/bandits/#VLQuantitativeFinancePackage.MyTickerPickerSIMRiskAwareWorldModel) to represent the world, both using [custom `build(...)` methods](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/bandits/#VLQuantitativeFinancePackage.build-Tuple{Type{MyTickerPickerSIMRiskAwareWorldModel},%20NamedTuple}).

Specify the list of tickers to examine in the `my_list_of_tickers::Vector{String}` array, and store the number of tickers in the `K::Int64` variable:

In [11]:
my_list_of_tickers, K = let
    
    my_tickers = ["AAPL", "MSFT", "ADBE", "MRK", "PFE", "JNJ", "MET", "NFLX", "AMD", "MU", "NVDA", "INTC", "MMM",
        "UNH", "JPM", "OXY", "TSLA", "PEP", "PG", "UPS", "COST", "TGT", "BAC", "C", "WFC", "KR", "WMT", "GS", "LLY", 
        "AMGN", "CVS", "ABT", "MDT", "HON", "CAT", "DE", "LMT", "BA", "XOM", "CVX"];

    unique!(my_tickers); # ensure uniqueness
    K = length(my_tickers);
    my_tickers, K
end;

__Check__: Are all the tickers in `my_list_of_tickers` present in the `list_of_tickers` variable defined earlier? If not, modify `my_list_of_tickers` accordingly.

In [12]:
let
    
    # check that all tickers are in our dataset
    for ticker ∈ my_list_of_tickers
        if !(ticker ∈ list_of_tickers)
            error("Ticker $ticker is not in the dataset.")
        end
    end
    println("All $(length(my_list_of_tickers)) tickers are valid and present in the dataset.")
end;

All 40 tickers are valid and present in the dataset.


Next, construct [a `MyEpsilonSamplingBanditModel` instance](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/bandits/#VLQuantitativeFinancePackage.MyEpsilonSamplingBanditModel), which holds information about the [ε-greedy sampling approach](https://arxiv.org/abs/1707.02038). The [`build(...)` method](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/bandits/#VLQuantitativeFinancePackage.build-Tuple{Type{MyEpsilonSamplingBanditModel},%20NamedTuple}) takes the model type and a [NamedTuple](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) holding the number of arms `K::Int64`, initial values for the `α::Vector{Int64}` and `β::Vector{Int64}` arrays (success and failure counters for each arm), and the exploration parameter `ϵ::Float64` controlling the exploration fraction. 

We save the model instance to the `bandit_model::MyEpsilonSamplingBanditModel` variable:

In [ ]:
bandit_model = build(MyEpsilonSamplingBanditModel, (
    K = K, # number bandit arms
    α = ones(K), # initialize to uniform values
    β = ones(K), # initialize to uniform values
));

### Setup a MyTickerPickerSIMRiskAwareWorldModel instance

Now that we have the bandit model, construct [a `MyTickerPickerSIMRiskAwareWorldModel` instance](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/bandits/#VLQuantitativeFinancePackage.MyTickerPickerSIMRiskAwareWorldModel) holding world data. The `world(...)` function takes the `start::Int64` index, an action $a\in\mathcal{A}$ value, and the `world_model::MyTickerPickerSIMRiskAwareWorldModel` instance, then returns a binary reward $r\in\left\{0,1\right\}$.

> __Reward Structure:__
>
> We compare the risk-adjusted excess growth rate of ticker $i$ against the mean market excess growth rate $\bar{G}_{m}$. If the ticker's risk-adjusted excess growth rate exceeds the market benchmark, we assign a reward of 1 (success); otherwise, we assign a reward of 0 (failure). The risk-adjustment factor is $\beta_{i}^{\lambda}$, where $\beta$ is the ticker's beta coefficient from the single index model, and $\lambda\in\mathbb{R}_{\geq{0}}$ is the risk-aversion parameter. Higher $\lambda$ values penalize high-beta (risky) assets more severely.

Let's implement the risk-aware world model:

In [ ]:
function world(start::Int64, action::Int64, worldmodel::MyTickerPickerSIMRiskAwareWorldModel)::Int64

    # get data from the world model -
    tickers = worldmodel.tickers; # what is my list of tickers?
    parameters = worldmodel.parameters; # single index model parameters
    risk_free_rate = worldmodel.risk_free_rate; # risk-free rate
    Δt = worldmodel.Δt; # time step
    Ḡₘ = worldmodel.Ḡₘ; # mean market growth rate
    buffersize = worldmodel.buffersize; # number of time steps to look ahead
    riskfactors = worldmodel.risk; # risk dictionary
    
    # initialize -
    result_flag = 0;

    # grab the ticker we are looking at
    ticker_symbol = tickers[action];
    RF = riskfactors[ticker_symbol]; # risk factor for this ticker

    # get the SIM parameters for this ticker -
    sim_params = parameters[ticker_symbol];
    αᵢ = sim_params.alpha; # asset i intercept
    βᵢ = sim_params.beta; # asset i sensitivity to market
    TV = sim_params.training_variance; # asset i total variance
    σᵢ = (1/RF)*sqrt(Δt*TV); # asset i standard deviation
    gᵢ = (αᵢ/RF) + (βᵢ/RF)*(Ḡₘ)+(σᵢ*randn()); # adjusted growth rate for asset i    

    # Does this ticker beat the market benchmark?
    (gᵢ > Ḡₘ) ? result_flag = 1 : result_flag = 0;
    return result_flag;
end;

Finally, construct a [`MyTickerPickerSIMRiskAwareWorldModel` instance](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/bandits/#VLQuantitativeFinancePackage.MyTickerPickerSIMRiskAwareWorldModel) using [a custom `build(...)` method](https://varnerlab.github.io/VLQuantitativeFinancePackage.jl/dev/bandits/#VLQuantitativeFinancePackage.build-Tuple{Type{MyTickerPickerSIMRiskAwareWorldModel},%20NamedTuple}). 

The method takes the model type and a [NamedTuple](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) with the `my_list_of_tickers::Vector{String}` array, the `data::Dict{String, DataFrame}` field holding price data, the `world::Function`, the `risk_free_rate::Float64`, the time-step $\Delta{t}$, and the `buffersize::Int64` field. 

We store the world model in the `my_world_model::MyTickerPickerSIMRiskAwareWorldModel` variable:

In [ ]:
my_world_model = let

    # build the risk dictionary -
    risk = Dict{String, Float64}();
    for ticker in my_list_of_tickers
        sim_params = sim_model_parameters[ticker];
        βᵢ = sim_params.beta; # asset i beta (sensitivity to market)
        risk[ticker] = βᵢ^(λ) ; # risk factor based on beta raised to the risk-aversion parameter
    end
    
    # build the world model -
    world_model = build(MyTickerPickerSIMRiskAwareWorldModel, (
        tickers = my_list_of_tickers,
        parameters = sim_model_parameters,
        world = world,
        Ḡₘ = Ḡₘ,
        risk = risk,
        risk_free_rate = risk_free_rate,
        Δt = Δt,
        buffersize = 0, # number of time steps to look ahead
    ));

    world_model; # return
end;

## Task 2: Implement the trade bot loop
Fill me in later.

In [ ]:
let
end;

## Summary
One direct summary sentence goes here. 

> __Key Takeaways:__
> 
> Three key takeaways go here.

One direct conclusion sentence goes here.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___